# WarpTracer benchmarks

Select **Runtime → Change runtime type → GPU**, then run setup, benchmark, and results.
Choose one preset: `scale`, `grid`, `oval`, `navigation`, `batches`, `fusion`, or `quick`.
Presets set car counts, workloads, durations and output paths; no other settings need editing.
The default `scale` compares mesh, uncached EDT and cached EDT with identical moving 3D sensor poses and distributed navigation starts, at 256 / 1024 / 4096 cars. It reports scan accuracy and cache speed ratios. No PPO inference or training is included. Choose `grid` to repeat the earlier fixed-scan comparison.
`oval` runs the mesh navigation baseline. `navigation` compares the room LiDAR baseline with disparity control. GPU presets stop immediately if CUDA is unavailable.
The `quick` preset also works on CPU. Output streams into the cell.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

repo = Path("/content/WarpTracer-rigidbody")
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--single-branch", "--branch",
                    "performance-prototype", "https://github.com/yuywe/WarpTracer.git", str(repo)], check=True)
os.chdir(repo)
subprocess.run(["git", "fetch", "origin", "performance-prototype:refs/remotes/origin/performance-prototype"], check=True)
if subprocess.run(["git", "show-ref", "--verify", "--quiet", "refs/heads/performance-prototype"]).returncode:
    subprocess.run(["git", "branch", "--no-track", "performance-prototype", "origin/performance-prototype"], check=True)
subprocess.run(["git", "switch", "performance-prototype"], check=True)
subprocess.run(["git", "pull", "--ff-only", "origin", "performance-prototype"], check=True)
os.environ["UV_PYTHON"] = sys.executable
subprocess.run(["uv", "sync", "--locked"], check=True)

def run_stream(command):
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1, start_new_session=True) as process:
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
            if process.wait():
                raise subprocess.CalledProcessError(process.returncode, command)
        except KeyboardInterrupt:
            import signal
            os.killpg(process.pid, signal.SIGTERM)
            process.wait()
            raise


In [ ]:
preset = "scale"  # or "grid", "oval", "navigation", "batches", "fusion", "quick"
run_stream(["uv", "run", "warptracer", "benchmark", preset])


In [ ]:
import json
report_path = Path("outputs/oval-benchmark.json" if preset == "oval" else f"outputs/{preset}.json")
report = json.loads(report_path.read_text())
print(report["device_name"], report["physics_engine"])
for result in report["results"]:
    s = result["summary"]
    print("cars:", result["environments"], result["case"], result.get("lidar_algorithm", result.get("lidar_backend", "mesh")),
          result["integrator"], result["backend"],
          "cached" if result.get("grid_sample_cache") else "uncached" if result.get("grid_sample_cache") is False else "",
          round(s["median_aggregate_environment_transitions_per_second"]), "aggregate transitions/s;",
          round(s["median_batch_transitions_per_second"]), "batch transitions/s;",
          "variable timing" if s["timing_variable"] else "max/min <= 1.2")
    march = [trial.get("march_diagnostics") for trial in result["trials"]]
    if any(march):
        print("  March iterations / exhaustion:",
              [(m["max_iterations_observed"], m["iteration_limit_events"]) for m in march if m])
for comparison in report.get("lidar_speedups", []):
    print("GRID/MESH cars:", comparison["environments"], comparison["case"],
          "cached" if comparison.get("grid_sample_cache") else "uncached",
          round(comparison["grid_over_mesh"], 3), "x (above 1 means grid is faster)")
for comparison in report.get("cache_speedups", []):
    print("CACHED/UNCACHED cars:", comparison["environments"], comparison["case"],
          round(comparison["cached_over_uncached"], 3), "x (above 1 means caching is faster)")
comparison = report.get("scan_comparison")
if comparison:
    print("Identical-pose range errors (meters):", comparison["absolute_range_error_m"])
    print("March diagnostics:", comparison.get("march_diagnostics"))
    print("Hit/miss differences:", comparison["valid_mismatch_count"], "/", comparison["rays"])
    print("Common-hit errors over 1 meter:", comparison["common_hit_errors_over_1m"])
print("Full report:", report_path.resolve())
